# Assignment 4
Disclaimer : Run it in standard CPU no T4 GPU \
 \
 \
 \
## Title - Parallel Computing Using CUDA
Problem Statement - Matrix Multiplication using CUDA C

This pip installation is used for the installation of the CUDA(Compute Unified Device Architecture) compiler which has the the NVCC (Nvedia cuda compiler) plugin in it

In [1]:
!pip install git+https://github.com/afnan47/cuda.git

  Cloning https://github.com/afnan47/cuda.git to /tmp/pip-req-build-cgitlmpp
  Running command git clone --filter=blob:none --quiet https://github.com/afnan47/cuda.git /tmp/pip-req-build-cgitlmpp
  Resolved https://github.com/afnan47/cuda.git to commit aac710a35f52bb78ab34d2e52517237941399eff
  Preparing metadata (setup.py) ... done


Now load the plugin

In [2]:
%load_ext nvcc_plugin

directory /content/src already exists
Out bin /content/result.out


# Part 1
Problem Statement -Addition of two large vectors

In [ ]:
%%writefile vector.cu
#include <iostream>    
#include <vector>      
#include <chrono>     // for time measuring 
#include <cuda.h> 

using namespace std;
using namespace std::chrono;

__global__ void add(int* A, int* B, int* C, int size) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < size) {
        C[tid] = A[tid] + B[tid];
    }
}

// Sequential CPU implementation of vector addition
void sequentialAddition(int* A, int* B, int* C, int size) {
    for (int i = 0; i < size; i++) {
        C[i] = A[i] + B[i];
        cout << C[i] << " " ;
    }
    cout <<endl ; 
}

int main() {
    int N;           
    cout << "Enter the size of the vectors: ";
    cin >> N;

    int* A, * B, * C;
    
    size_t vectorBytes = N * sizeof(int);

    A = new int[N];
    B = new int[N];
    C = new int[N];

    // inserting default value here you can take proper custom value 
    for (int i = 0; i < N; i++) {
        A[i] = i + 1;
        B[i] = i * 2 ; 
    }

    // Print the input vectors
    for (int i = 0; i < N; i++) {
        cout << A[i] << " ";
    }
    cout << endl;
    for (int i = 0; i < N; i++) {
        cout << B[i] << " ";
    }
    cout << endl;

    int* X, * Y, * Z;

    cudaMalloc(&X, vectorBytes);
    cudaMalloc(&Y, vectorBytes);
    cudaMalloc(&Z, vectorBytes);

    cudaMemcpy(X, A, vectorBytes, cudaMemcpyHostToDevice);
    cudaMemcpy(Y, B, vectorBytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    auto start = high_resolution_clock::now();
    sequentialAddition(A, B, C, N);
    auto stop = high_resolution_clock::now();
    auto seq_duration = duration_cast<microseconds>(stop - start);
    cout << "Sequential Addition Time: " << seq_duration.count() << " microseconds" << endl;

    start = high_resolution_clock::now();
    add<<<blocksPerGrid, threadsPerBlock>>>(X, Y, Z, N);
    cudaMemcpy(C, Z, vectorBytes, cudaMemcpyDeviceToHost);
    for (int i = 0 ; i <  N ; i ++){
        cout << C[i] << " " ; 
    }
    cout << endl ; 
    stop = high_resolution_clock::now();
    auto par_duration = duration_cast<microseconds>(stop - start);
    cout << "Parallel Addition Time: " << par_duration.count() << " microseconds" << endl;

    delete[] A;
    delete[] B;
    delete[] C;

    cudaFree(X);
    cudaFree(Y);
    cudaFree(Z);

    return 0;
}

Writing vector.cu


In [16]:
!nvcc vector.cu -o vector
!./vector

Use default vector size (8) and elements? (y/n): y
Initializing Vector A:
Initializing Vector B:
Vector A: 1 2 3 4 5 6 7 8 
Vector B: 1 2 3 4 5 6 7 8 
Sequential Addition: 2 4 6 8 10 12 14 16 
Parallel Addition: 2 4 6 8 10 12 14 16 
Sequential Addition Time: 0 microseconds
Parallel Addition Time: 15 microseconds


# Part 2
Problem Statement - Matrix Multiplication using CUDA C

do not remove the %% cu it is the identidfer for the cuda code

In [ ]:
%%writefile matrix.cu
#include <iostream>                                                             // For input and output operations
#include <cuda.h>                                                               // CUDA runtime library
#include <chrono>                                                               // For measuring execution time

using namespace std;
using namespace std::chrono;

                                                                                // CUDA kernel for matrix multiplication
__global__ void multiply(int* A, int* B, int* C, int M, int N, int K) {
                                                                                // Calculate the row index for the current thread
    int row = blockIdx.y * blockDim.y + threadIdx.y;
                                                                                // Calculate the column index for the current thread
    int col = blockIdx.x * blockDim.x + threadIdx.x;

                                                                                // Check if the current thread is within the bounds of the output matrix
    if (row < M && col < K) {
        int sum = 0;
                                                                                // Iterate through the elements of the rows of A and columns of B
        for (int i = 0; i < N; i++) {
                                                                                // Perform the multiplication and accumulate the sum
            sum += A[row * N + i] * B[i * K + col];
        }
                                                                                // Store the result in the corresponding element of the output matrix C
        C[row * K + col] = sum;
    }
}

                                                                                // Function to initialize a matrix with default values or user input
void initialize(int* matrix, int rows, int cols, bool useDefault = false) {
    if (useDefault) {
                                                                                // Default initialization for 4x4 matrices
        for (int i = 0; i < rows * cols; i++) {
            matrix[i] = i + 1;  // Simple pattern: 1, 2, 3, ...
        }
    } else {
                                                                                // Manual input from the user
        for (int i = 0; i < rows * cols; i++) {
            cout << "Enter element " << i + 1 << ": ";
            cin >> matrix[i];
        }
    }
}

                                                                                // Function to print the elements of a matrix
void print(int* matrix, int rows, int cols) {
    for (int row = 0; row < rows; row++) {
        for (int col = 0; col < cols; col++) {
            cout << matrix[row * cols + col] << " ";
        }
        cout << '\n';                                                           // Move to the next row after printing all columns
    }
    cout << '\n';                                                               // Add an extra newline for better readability
}

                                                                                // Sequential CPU implementation of matrix multiplication
void sequentialMultiply(int* A, int* B, int* C, int M, int N, int K) {
    for (int i = 0; i < M; i++) {                                               // Iterate through rows of A
        for (int j = 0; j < K; j++) {                                           // Iterate through columns of B (and rows of C)
            int sum = 0;
            for (int k = 0; k < N; k++) {                                       // Iterate through columns of A (and rows of B)
                sum += A[i * N + k] * B[k * K + j];
            }
            C[i * K + j] = sum;                                                 // Store the result in C
        }
    }
}

int main() {
    int M, N, K;
    char choice;
    bool useDefault = false;

    cout << "Use default 4x4 matrices? (y/n): ";
    cin >> choice;

    if (choice == 'y' || choice == 'Y') {
        M = 4;
        N = 4;
        K = 4;
        useDefault = true;
    } else {
        cout << "Enter the number of rows of the first matrix: ";
        cin >> M;
        cout << "Enter the number of columns of the first matrix (and rows of the second): ";
        cin >> N;
        cout << "Enter the number of columns of the second matrix: ";
        cin >> K;
    }

                                                                                // Allocate memory on the host (CPU)
    int* A = new int[M * N];
    int* B = new int[N * K];
    int* C = new int[M * K];

                                                                                // Initialize the matrices A and B
    cout << "Initializing Matrix A:\n";
    initialize(A, M, N, useDefault);

    cout << "Initializing Matrix B:\n";
    initialize(B, N, K, useDefault);

                                                                                // Print the input matrices
    cout << "Matrix A: \n";
    print(A, M, N);

    cout << "Matrix B: \n";
    print(B, N, K);

                                                                                // Allocate memory on the device (GPU)
    int* X, * Y, * Z;
    cudaMalloc(&X, M * N * sizeof(int));
    cudaMalloc(&Y, N * K * sizeof(int));
    cudaMalloc(&Z, M * K * sizeof(int));

                                                                                // Copy matrices A and B from host to device
    cudaMemcpy(X, A, M * N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(Y, B, N * K * sizeof(int), cudaMemcpyHostToDevice);

                                                                                // Define the number of threads per block
    int THREADS = 16;

                                                                                // Calculate the number of blocks needed in each dimension
    int BLOCKS_X = (K + THREADS - 1) / THREADS;
    int BLOCKS_Y = (M + THREADS - 1) / THREADS;

                                                                                // Define the dimensions of the thread block and the grid of blocks
    dim3 threads(THREADS, THREADS);
    dim3 blocks(BLOCKS_X, BLOCKS_Y);

                                                                                // Perform sequential matrix multiplication on the CPU
    auto start = high_resolution_clock::now();
    sequentialMultiply(A, B, C, M, N, K);
    auto stop = high_resolution_clock::now();
    auto seq_duration = duration_cast<microseconds>(stop - start);

    cout << "Sequential Multiplication of matrix A and B: \n";
    print(C, M, K);

                                                                                // Perform parallel matrix multiplication on the GPU
    start = high_resolution_clock::now();
    multiply<<<blocks, threads>>>(X, Y, Z, M, N, K);
    cudaDeviceSynchronize(); // Wait for the GPU kernel to complete
                                                                                // Copy the result matrix C from device to host
    cudaMemcpy(C, Z, M * K * sizeof(int), cudaMemcpyDeviceToHost);
    stop = high_resolution_clock::now();
    auto par_duration = duration_cast<microseconds>(stop - start);

    cout << "Parallel Multiplication of matrix A and B: \n";
    print(C, M, K);

                                                                                // Print the execution times and the speedup
    cout << "Sequential Multiplication Time: " << seq_duration.count() << " microseconds" << endl;
    cout << "Parallel Multiplication Time: " << par_duration.count() << " microseconds" << endl;
    cout << "Speedup: " << (float)seq_duration.count() / par_duration.count() << "x" << endl;

                                                                                // Free the allocated memory on the host
    delete[] A;
    delete[] B;
    delete[] C;

                                                                                // Free the allocated memory on the device
    cudaFree(X);
    cudaFree(Y);
    cudaFree(Z);

    return 0;
}

Overwriting matrix.cu


In [11]:
!nvcc matrix.cu -o matrix
!./matrix

Use default 4x4 matrices? (y/n): y
Initializing Matrix A:
Initializing Matrix B:
Matrix A: 
1 2 3 4 
5 6 7 8 
9 10 11 12 
13 14 15 16 

Matrix B: 
1 2 3 4 
5 6 7 8 
9 10 11 12 
13 14 15 16 

Sequential Multiplication of matrix A and B: 
90 100 110 120 
202 228 254 280 
314 356 398 440 
426 484 542 600 

Parallel Multiplication of matrix A and B: 
90 100 110 120 
202 228 254 280 
314 356 398 440 
426 484 542 600 

Sequential Multiplication Time: 0 microseconds
Parallel Multiplication Time: 3 microseconds
Speedup: 0x
